# Clarius RF Data Reader (NumPy 2.4.2 Compatible)

This notebook reads and analyzes raw RF data from Clarius ultrasound scanners.

## File Format

According to the Clarius documentation, the raw file structure is:

```
Header:
  uint32 id
  uint32 numFrames
  uint32 numScanLines
  uint32 numSamplesPerLine
  uint32 sampleSizeInBytes

For each frame:
  uint64 timestamp (in nanoseconds)
  Data block: numScanLines × numSamplesPerLine × sampleSizeInBytes
```

RF data is always 16-bit beamformed samples.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import struct

print(f"NumPy version: {np.__version__}")

# File path
rf_file = '/mnt/user-data/uploads/2026-01-17T00-04-13_0000_rf.raw'

Fontconfig warning: ignoring UTF-8: not a valid region tag
Matplotlib is building the font cache; this may take a moment.


NumPy version: 1.26.4


## Step 1: Read the Header

In [ ]:
def read_header(filename):
    """
    Read the header from Clarius raw data file.
    
    Returns:
        dict: Header information containing id, numFrames, numScanLines, 
              numSamplesPerLine, and sampleSizeInBytes
    """
    with open(filename, 'rb') as f:
        # Read 5 uint32 values (4 bytes each = 20 bytes total)
        header_data = f.read(20)
        
        # Unpack as little-endian unsigned 32-bit integers
        header = struct.unpack('<5I', header_data)
        
        header_dict = {
            'id': header[0],
            'numFrames': header[1],
            'numScanLines': header[2],
            'numSamplesPerLine': header[3],
            'sampleSizeInBytes': header[4]
        }
        
    return header_dict

# Read and display header
header = read_header(rf_file)
print("Header Information:")
print("="*50)
for key, value in header.items():
    print(f"{key:20s}: {value}")
print("="*50)

# Calculate expected file size
header_size = 20
timestamp_size = 8  # uint64
frame_data_size = header['numScanLines'] * header['numSamplesPerLine'] * header['sampleSizeInBytes']
expected_size = header_size + header['numFrames'] * (timestamp_size + frame_data_size)

print(f"\nFrame data size: {frame_data_size:,} bytes")
print(f"Expected file size: {expected_size:,} bytes")

import os
actual_size = os.path.getsize(rf_file)
print(f"Actual file size: {actual_size:,} bytes")
print(f"Match: {expected_size == actual_size}")

## Step 2: Read All Frames

In [ ]:
def read_rf_data(filename):
    """
    Read all RF data from Clarius raw data file.
    NumPy 2.4.2 compatible version.
    
    Returns:
        header: Header dictionary
        timestamps: Array of timestamps (in nanoseconds)
        rf_data: 3D numpy array [frames, scanlines, samples]
    """
    # Read header
    header = read_header(filename)
    
    # Initialize arrays with explicit dtypes
    timestamps = np.zeros(header['numFrames'], dtype=np.uint64)
    rf_data = np.zeros((header['numFrames'], 
                        header['numScanLines'], 
                        header['numSamplesPerLine']), 
                       dtype=np.int16)
    
    with open(filename, 'rb') as f:
        # Skip header
        f.seek(20)
        
        # Read each frame
        for frame_idx in range(header['numFrames']):
            # Read timestamp (uint64, 8 bytes)
            timestamp_data = f.read(8)
            timestamps[frame_idx] = struct.unpack('<Q', timestamp_data)[0]
            
            # Read frame data
            # NumPy 2.x: Use string dtype format for explicit byte order
            frame_size = header['numScanLines'] * header['numSamplesPerLine']
            frame_data = np.fromfile(f, dtype='<i2', count=frame_size)
            
            # Ensure we got the expected amount of data
            if len(frame_data) != frame_size:
                raise ValueError(f"Expected {frame_size} samples, got {len(frame_data)}")
            
            # Reshape to [scanlines, samples] - explicit tuple for NumPy 2.x
            rf_data[frame_idx] = frame_data.reshape((header['numScanLines'], 
                                                     header['numSamplesPerLine']))
    
    return header, timestamps, rf_data

# Read all data
print("Reading RF data...")
header, timestamps, rf_data = read_rf_data(rf_file)
print(f"Successfully read {header['numFrames']} frames")
print(f"RF data shape: {rf_data.shape} [frames, scanlines, samples]")
print(f"Data type: {rf_data.dtype}")
print(f"Data range: [{np.min(rf_data)}, {np.max(rf_data)}]")

## Step 3: Display Timestamps

In [ ]:
print("Timestamps (nanoseconds):")
print("="*50)
for i, ts in enumerate(timestamps):
    print(f"Frame {i}: {ts}")

if len(timestamps) > 1:
    # Calculate frame intervals
    intervals = np.diff(timestamps)
    print(f"\nFrame intervals (ms):")
    for i, interval in enumerate(intervals):
        print(f"Frame {i} to {i+1}: {interval/1e6:.2f} ms")
    print(f"\nMean interval: {np.mean(intervals)/1e6:.2f} ms")
    print(f"Frame rate: {1e9/np.mean(intervals):.2f} Hz")

## Step 4: Visualize RF Data

In [ ]:
# Plot first frame
frame_to_plot = 0

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Raw RF data (all scan lines)
im1 = axes[0].imshow(rf_data[frame_to_plot].T, 
                     aspect='auto', 
                     cmap='gray',
                     extent=[0, header['numScanLines'], 
                            header['numSamplesPerLine'], 0])
axes[0].set_xlabel('Scan Line')
axes[0].set_ylabel('Sample (Depth)')
axes[0].set_title(f'Raw RF Data - Frame {frame_to_plot}')
plt.colorbar(im1, ax=axes[0], label='Amplitude')

# Plot 2: Single A-line (middle scan line)
middle_line = header['numScanLines'] // 2
axes[1].plot(rf_data[frame_to_plot, middle_line, :])
axes[1].set_xlabel('Sample Number')
axes[1].set_ylabel('RF Amplitude')
axes[1].set_title(f'Single A-line (Scan Line {middle_line})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Calculate Envelope (for attenuation analysis)

In [ ]:
from scipy.signal import hilbert

def calculate_envelope(rf_signal):
    """
    Calculate envelope of RF signal using Hilbert transform.
    
    Args:
        rf_signal: 1D RF signal array
    
    Returns:
        envelope: Amplitude envelope
    """
    # Convert to float for processing
    rf_float = rf_signal.astype(np.float64)
    analytic_signal = hilbert(rf_float)
    envelope = np.abs(analytic_signal)
    return envelope

# Calculate envelope for middle A-line
middle_line = header['numScanLines'] // 2
rf_aline = rf_data[0, middle_line, :]
envelope = calculate_envelope(rf_aline)

# Plot RF signal and envelope
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(rf_aline, label='RF Signal', alpha=0.7)
ax.plot(envelope, label='Envelope', linewidth=2, color='red')
ax.set_xlabel('Sample Number (proportional to depth)')
ax.set_ylabel('Amplitude')
ax.set_title('RF Signal and Envelope')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Step 6: Convert to dB for Attenuation Analysis

In [ ]:
# Convert envelope to dB
# Add small epsilon to avoid log(0)
epsilon = np.finfo(np.float64).eps  # NumPy 2.x: Use finfo for machine epsilon
envelope_dB = 20.0 * np.log10(envelope + epsilon)

# Plot envelope in dB
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(envelope_dB)
ax.set_xlabel('Sample Number (proportional to depth)')
ax.set_ylabel('Amplitude (dB)')
ax.set_title('Envelope in dB - Ready for TGC Compensation')
ax.grid(True, alpha=0.3)
plt.show()

print(f"Envelope dB range: [{np.min(envelope_dB):.2f}, {np.max(envelope_dB):.2f}] dB")

## Step 7: Average Multiple A-lines to Reduce Speckle

In [ ]:
# Average all scan lines in first frame
# NumPy 2.x: Explicit axis specification
averaged_rf = np.mean(rf_data[0], axis=0, dtype=np.float64)
averaged_envelope = calculate_envelope(averaged_rf)
averaged_envelope_dB = 20.0 * np.log10(averaged_envelope + epsilon)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Single A-line
axes[0].plot(envelope_dB, alpha=0.7, label='Single A-line')
axes[0].set_xlabel('Sample Number')
axes[0].set_ylabel('Amplitude (dB)')
axes[0].set_title('Single A-line (noisy due to speckle)')
axes[0].grid(True, alpha=0.3)

# Averaged A-lines
axes[1].plot(averaged_envelope_dB, color='red', linewidth=2, label='Averaged')
axes[1].set_xlabel('Sample Number')
axes[1].set_ylabel('Amplitude (dB)')
axes[1].set_title(f'Averaged over {header["numScanLines"]} A-lines (smoother)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Export Data for Further Analysis

In [ ]:
# Export averaged envelope data to CSV for external analysis
output_data = np.column_stack([
    np.arange(len(averaged_envelope)),  # Sample number
    averaged_envelope,                   # Linear amplitude
    averaged_envelope_dB                 # dB amplitude
])

# Save with headers
np.savetxt('/mnt/user-data/outputs/averaged_envelope.csv', 
           output_data,
           delimiter=',',
           header='Sample,Amplitude_Linear,Amplitude_dB',
           comments='',
           fmt=['%d', '%.6f', '%.6f'])

print("Exported averaged envelope data to: averaged_envelope.csv")
print(f"Data shape: {output_data.shape}")

## Summary

This notebook successfully:
1. Read the Clarius RF raw data header (NumPy 2.4.2 compatible)
2. Extracted timestamps and RF data for all frames
3. Visualized the raw RF data
4. Calculated the envelope using Hilbert transform
5. Converted to dB scale for attenuation analysis
6. Demonstrated averaging to reduce speckle noise
7. Exported processed data for further analysis

### Key NumPy 2.4.2 Updates:
- Used string dtype format `'<i2'` instead of `np.int16` for `fromfile()`
- Explicit tuple for `reshape()` dimensions
- Used `np.finfo().eps` for machine epsilon
- Explicit `dtype` in aggregation functions like `mean()`
- Explicit `axis` parameter specifications

### Next Steps for Attenuation Coefficient Calculation:

1. **Load the corresponding .tgc.yml file** to get TGC gain values
2. **Load the .yml metadata file** to convert sample numbers to actual depth (mm/cm)
3. **Interpolate TGC gains** for each depth/sample
4. **Subtract TGC from dB envelope**: `corrected_dB = envelope_dB - tgc_gain_dB`
5. **Fit linear regression** to corrected_dB vs depth
6. **Extract slope** as attenuation coefficient (dB/cm)